# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the Croissant package specification.

### Dataset Source
Croissant JSON-LD schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Install the mlcroissant package (if not installed)
!pip install mlcroissant --quiet

## 1. Data Loading

Load dataset metadata via Croissant schema and inspect the package using `mlcroissant`. The `Dataset` object will provide both the metadata and access to records in the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# -- Define the Croissant schema URL --
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# -- Load the dataset metadata --
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview

Examine the available record sets, their `@id`s, and the fields and columns within each record set.

Use the Croissant schema to list all entities by their unique `@id` as required.

In [ ]:
# List all record sets by @id and display their fields and columns by @id
for rs in metadata.record_sets:
    print(f"\nRecord set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    # List the fields in the record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} ({field.name}, type: {field.data_type})")
    # List the columns (if present)
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.id} ({col.name}, type: {col.data_type})")

## 3. Data Extraction

Load the records from each record set into a Pandas DataFrame. Use the `@id` of each record set as provided above, and reference all columns or fields via their respective `@id`s.

In [ ]:
#-- Retrieve record set @ids --#
record_set_ids = [rs.id for rs in metadata.record_sets]

#-- Extract all record sets into dataframes --#
dataframes = {}
for record_set_id in record_set_ids:
    # Fetch all records from this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print('Loaded DataFrames for record_set_ids:', record_set_ids)

# Preview the columns of the first record set and show the first few rows
if record_set_ids:
    print('Columns:', dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a relevant numeric field (referenced by its `@id`) from the main patient dataset and perform common EDA operations:
- Filter records above a certain threshold
- Normalize values
- Group by a key field

**Note:** You should replace `YOUR_NUMERIC_FIELD_ID` and `YOUR_GROUP_FIELD_ID` with the actual `@id` values from the overview cells above. If unsure, review the output from Section 2.

In [ ]:
# Example EDA: Filter, normalize, and group by field

# Choose the first record set (assuming it's the main dataset) and inspect its columns
main_rs_id = record_set_ids[0]
main_df = dataframes[main_rs_id]

# For demonstration, let's try to select a numeric field (@id) for filtering and normalization
# Example: suppose field '@id': 'cr:PatientAge' and '@id': 'cr:Sex' exist
# Replace with real @id's! (Review Section 2 output)
numeric_field_id = None
group_field_id = None

# Try to automatically guess numeric and group fields
for col in main_df.columns:
    if numeric_field_id is None and ("age" in col.lower()):
        numeric_field_id = col
    if group_field_id is None and ("sex" in col.lower() or "gender" in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    # Select the first numeric column (float or int) if "age" is not present
    numeric_cols = main_df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]

if group_field_id is None:
    # Try to use first object/categorical column
    group_cols = main_df.select_dtypes(include=['object', 'category']).columns.tolist()
    if group_cols:
        group_field_id = group_cols[0]

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Filter records for values > threshold (e.g., age > 50)
if numeric_field_id and numeric_field_id in main_df.columns:
    threshold = 50
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")

    # Normalize the numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field and see mean values
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename({numeric_field_id: f"mean_{numeric_field_id}"}, axis=1)
        )
        print(f"Grouped data by {group_field_id} and mean {numeric_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric_field_id found for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and plot group comparisons if possible.

**All field references must be by `@id` (column names in the DataFrame).**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field (if available)
if numeric_field_id and numeric_field_id in main_df.columns:
    sns.histplot(main_df[numeric_field_id].dropna(), bins=12)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Compare distribution by group
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

- We demonstrated loading and exploring a Croissant dataset using the `mlcroissant` library referencing all entities by their `@id`.
- Main findings: the dataset contains detailed clinicopathologic variables on 77 cancer survivors with second primary colorectal cancer.
- Our exploratory analysis illustrated basic filtering, normalization, grouping, and visualization of numeric fields by `@id`.

See the FAIR² [dataset landing page](https://sen.science/doi/10.71728/senscience.qs2f-h81p) for further metadata and documentation.
